# Assignment 7 — Support Vector Regression for Final Grade (G3) Prediction
**Dataset:** UCI Student Performance (`student-por.csv`)  
**Target:** continuous `G3`; inputs exclude G1/G2/G3

**Aim:** Tune SVR (kernel, C, epsilon, gamma) with GridSearchCV inside a scaling pipeline, report best params + MAE/RMSE/R², plot actual-vs-predicted, and compare against Linear Regression.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print(df["G3"].describe())
print("Missing:", int(df.isnull().sum().sum()))

count    649.000000
mean      11.906009
std        3.230656
min        0.000000
25%       10.000000
50%       12.000000
75%       14.000000
max       19.000000
Name: G3, dtype: float64
Missing: 0


## 4. Data preprocessing — StandardScaler inside pipeline

In [5]:
X_raw = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"].to_numpy()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train {X_train_raw.shape} Test {X_test_raw.shape}")

Train (519, 30) Test (130, 30)


C:\Users\ns479\AppData\Local\Temp\ipykernel_24716\239155765.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. Model implementation — SVR + GridSearchCV

In [6]:
svr_pipe = Pipeline([("prep", preprocess), ("svr", SVR())])
param_grid = {
    "svr__kernel": ["rbf", "linear"],
    "svr__C": [0.5, 1.0, 5.0],
    "svr__epsilon": [0.1, 0.5],
    "svr__gamma": ["scale", "auto"],
}
gs = GridSearchCV(svr_pipe, param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
gs.fit(X_train_raw, y_train)
print("Best params:", gs.best_params_)
print(f"Best CV RMSE: {np.sqrt(-gs.best_score_):.4f}")
pd.DataFrame([{"param": k, "value": str(v)} for k, v in gs.best_params_.items()]).to_csv("outputs/best_params.csv", index=False)

Best params: {'svr__C': 5.0, 'svr__epsilon': 0.5, 'svr__gamma': 'auto', 'svr__kernel': 'rbf'}
Best CV RMSE: 2.6767


## 6. Evaluation — MAE, RMSE, R² (SVR vs Linear Regression)

In [7]:
def scores(name, model):
    p = model.predict(X_test_raw)
    mae = mean_absolute_error(y_test, p); mse = mean_squared_error(y_test, p)
    r2 = r2_score(y_test, p)
    print(f"{name}: MAE={mae:.4f} RMSE={np.sqrt(mse):.4f} R2={r2:.4f}")
    return {"model": name, "MAE": mae, "RMSE": float(np.sqrt(mse)), "R2": r2, "pred": p}
lr = Pipeline([("prep", preprocess), ("lr", LinearRegression())]).fit(X_train_raw, y_train)
r_svr = scores("SVR-tuned", gs.best_estimator_)
r_lr = scores("LinearRegression", lr)
comp = pd.DataFrame([{k: v for k, v in r.items() if k != "pred"} for r in [r_svr, r_lr]])
print(comp.round(4).to_string(index=False))
comp.round(4).to_csv("outputs/comparison.csv", index=False)
svr_pred, lr_pred = r_svr["pred"], r_lr["pred"]

SVR-tuned: MAE=2.0039 RMSE=2.7715 R2=0.2123
LinearRegression: MAE=2.1564 RMSE=2.8618 R2=0.1602
           model    MAE   RMSE     R2
       SVR-tuned 2.0039 2.7715 0.2123
LinearRegression 2.1564 2.8618 0.1602


## 7. Tables and graphs — actual vs predicted + comparison

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, pred, title in zip(axes, [svr_pred, lr_pred], ["SVR (tuned)", "Linear Regression"]):
    ax.scatter(y_test, pred, alpha=0.55, s=20)
    lo, hi = float(min(y_test.min(), pred.min())), float(max(y_test.max(), pred.max()))
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_xlabel("Actual G3"); ax.set_ylabel("Predicted G3"); ax.set_title(title)
plt.suptitle("Actual vs predicted: SVR vs Linear Regression")
plt.tight_layout(); plt.savefig("outputs/actual_vs_pred.png", dpi=150); plt.close()
comp_melt = comp.melt(id_vars="model", var_name="metric", value_name="value")
plt.figure(figsize=(8, 5))
sns.barplot(data=comp_melt[comp_melt.metric.isin(["MAE", "RMSE"])], x="metric", y="value", hue="model")
plt.title("Error comparison (lower is better)")
plt.tight_layout(); plt.savefig("outputs/comparison.png", dpi=150); plt.close()
print("Saved: outputs/actual_vs_pred.png, outputs/comparison.png")

Saved: outputs/actual_vs_pred.png, outputs/comparison.png


## 8. Result and conclusion
- `StandardScaler` inside the pipeline; `GridSearchCV(cv=5)` over kernel/C/epsilon/gamma.
- Best params, MAE/RMSE/R² and SVR-vs-LR comparison are computed at runtime and saved to `outputs/`.
